In [ ]:
import time
import pandas as pd
import requests



In [ ]:
API_URL = "https://api.pcos.org.cn/pcos/app/web/analysis"

In [ ]:
HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json;charset=UTF-8",
    "Origin": "https://www.pcos.org.cn",
    "Referer": "https://www.pcos.org.cn/",
    "User-Agent": "Mozilla/5.0"
}

In [ ]:
INPUT_FILE = "dataset.xlsx"
OUTPUT_FILE = "pcos_predictions.xlsx"

print(INPUT_FILE)

In [ ]:
df = pd.read_excel(INPUT_FILE)

print(df.columns.tolist())
print(df.head())

In [ ]:
COLUMN_MAP = {
    "height": "height",
    "weight": "weight",
    "bmi": "BMI",
    "lh": "lh",
    "fsh": "fsh",
    "testosterone": "testosterone",
    "shbg": "shbg",
    "amh": "amh",
    "dheas": "dheas",
    "fasting_blood_glucose": "fasting_blood_glucose",
    "fasting_insulin": "fasting_insulin",
    "hdl": "hdl"
}

In [ ]:
def clean_value(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

results = []

In [ ]:
for _, row in df.iterrows():
    payload = {
        api_field: clean_value(row[file_col])
        for api_field, file_col in COLUMN_MAP.items()
    }

    try:
        response = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()

        result_row = {
            "api_code": data.get("code"),
            "api_msg": data.get("msg"),
            "predicted_cluster": data.get("data", {}).get("predicted_cluster"),
            "api_id": data.get("data", {}).get("id")
        }

    except Exception as e:
        result_row = {
            "api_code": None,
            "api_msg": f"ERROR: {e}",
            "predicted_cluster": None,
            "api_id": None
        }

    results.append(result_row)
    time.sleep(0.5)

In [ ]:
result_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
print(result_df)

In [ ]:
result_df.to_excel(OUTPUT_FILE, index=False)

print(f"Done. Saved to {OUTPUT_FILE}")

In [ ]:
time.sleep(1.0)

In [ ]:
result_df.head()